[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/prefect-certified/notebooks/day-01-intro-prefect.ipynb#scrollTo=a1b2c3d4)

---
# Day 1 · Introduction to Prefect: From Script to Workflow
**certified-journeys / prefect-certified** · Day 1 · Learn

> **Goal for today:** Convert a plain Python script into a Prefect flow, run it locally, and observe its state in the Prefect UI.


In [ ]:
%pip install -q prefect


## Step 1 · What is Prefect and why use it?

Prefect is a workflow orchestration framework that wraps ordinary Python functions with scheduling, retries, observability, and persistent state tracking.

| Plain Python script | Prefect workflow |
|---|---|
| Runs and disappears | Every run is persisted with state |
| Silent failures | Failed runs are visible in the UI |
| No retries out of the box | Configurable retries per task |
| Hard to schedule | Built-in scheduling, cron, intervals |
| No lineage | Full run history and logs |

Prefect's core concept is simple: a **flow** is a Python function decorated with `@flow`. That's it. Your existing code barely changes.


In [ ]:
# Verify Prefect installed correctly and check the version
import prefect

print(f"Prefect version: {prefect.__version__}")
print("Installation successful!")


### What just happened?

- **`import prefect`** loads the library — no server connection required for local runs.
- Prefect 2.x+ ("Prefect 2") is a complete rewrite of Prefect 1. It is the version taught here.
- **Prefect 3.x** (latest stable) maintains the same `@flow` / `@task` API — code in this course works on both 2.x and 3.x.
- In Colab you may see dependency resolution messages — these are normal and don't affect functionality.


## Step 2 · A plain Python function — before Prefect

Before adding Prefect, let's look at what we're converting. A typical ETL script might look like this: fetch data, process it, save a result. Right now it's just plain Python — no state tracking, no retries, no observability.

This is the "before" state. We'll transform it in the next step.


In [ ]:
import json
import urllib.request
from datetime import datetime

# --- Plain Python version (no Prefect) ---
# This mimics a typical ETL script: fetch → filter → summarise

def fetch_posts():
    """Fetch sample posts from a free public API."""
    # JSONPlaceholder is a free mock REST API — no key needed
    # Production equivalent: your data source (S3, database, API)
    url = "https://jsonplaceholder.typicode.com/posts?_limit=10"
    with urllib.request.urlopen(url) as response:
        return json.loads(response.read())

def filter_posts(posts, user_id=1):
    """Keep only posts belonging to the given user."""
    return [p for p in posts if p["userId"] == user_id]

def summarise(posts):
    """Return a short summary dict."""
    return {
        "count": len(posts),
        "titles": [p["title"][:40] for p in posts],
        "ran_at": datetime.utcnow().isoformat(),
    }

# Run it — no state, no retries, silent if it fails
posts = fetch_posts()
filtered = filter_posts(posts, user_id=1)
result = summarise(filtered)

print(f"Found {result['count']} posts for user 1")
for title in result['titles']:
    print(f"  • {title}")


### What just happened?

- This is a **real HTTP call** to JSONPlaceholder — a free, always-on mock API. No credentials needed.
- The script works, but if it fails mid-way you have **no record** of what succeeded or failed.
- **No automatic retries** if the network hiccups — the whole thing just crashes.
- **Nothing is observable** — you can't search past runs, filter by status, or see how long each step took.
- These are the exact pain points Prefect solves with a single decorator.


## Step 3 · Adding the `@flow` decorator

The minimum change to get Prefect observability is adding `@flow` to your top-level function. That's the entire migration for the basic case.

```python
from prefect import flow

@flow
def my_pipeline():
    ...
```

When you call `my_pipeline()`, Prefect:
1. Creates a **flow run** object with a unique ID
2. Sets its state to `Running`
3. Records any exception and sets state to `Failed`
4. Sets state to `Completed` on success
5. Writes structured logs you can query later


In [ ]:
from prefect import flow, get_run_logger
import json
import urllib.request
from datetime import datetime

# --- Prefect version: only change is @flow + logger ---

@flow(name="posts-etl-pipeline")  # name appears in the UI
def posts_etl_pipeline(user_id: int = 1):
    """Fetch, filter, and summarise posts — now with Prefect state tracking."""
    logger = get_run_logger()  # structured logger tied to this specific run

    logger.info(f"Starting pipeline for user_id={user_id}")

    # Fetch
    url = "https://jsonplaceholder.typicode.com/posts?_limit=10"
    with urllib.request.urlopen(url) as response:
        all_posts = json.loads(response.read())
    logger.info(f"Fetched {len(all_posts)} total posts")

    # Filter
    filtered = [p for p in all_posts if p["userId"] == user_id]
    logger.info(f"Filtered down to {len(filtered)} posts for user {user_id}")

    # Summarise
    result = {
        "count": len(filtered),
        "titles": [p["title"][:40] for p in filtered],
        "ran_at": datetime.utcnow().isoformat(),
    }

    logger.info(f"Pipeline complete — found {result['count']} posts")
    return result


# Running the flow is identical to calling any Python function
result = posts_etl_pipeline(user_id=1)
print(f"\nRun completed at: {result['ran_at']}")
print(f"Posts found: {result['count']}")


### What just happened?

- **`@flow`** is the only meaningful change — the business logic is identical.
- Prefect printed `Starting 'posts-etl-pipeline' run ...` — that's automatic, no extra code needed.
- **`get_run_logger()`** returns a logger that attaches each message to this specific flow run. If you run the flow 100 times, each run has its own log stream.
- **State tracking happens automatically** — if `urllib.request.urlopen` raised an exception, Prefect would catch it, mark the run `Failed`, and preserve the traceback.
- The flow returned its value normally — Prefect doesn't change the return contract.


## Step 4 · Flow run states and what they mean

Every Prefect flow run transitions through states. Understanding states is key to debugging and monitoring.

| State | Meaning |
|---|---|
| `Pending` | Scheduled but not yet executing |
| `Running` | Currently executing |
| `Completed` | Finished successfully |
| `Failed` | Raised an unhandled exception |
| `Crashed` | Infrastructure failure (OOM, signal) |
| `Cancelled` | Manually cancelled before completion |
| `Paused` | Waiting for human input |

You can inspect the state object returned by a flow call.


In [ ]:
from prefect import flow
from prefect.states import State

# return_state=True gives you the State object instead of the raw return value
@flow(name="state-demo")
def always_succeeds(x: int) -> str:
    return f"processed {x}"

@flow(name="failure-demo")
def sometimes_fails(x: int) -> str:
    if x < 0:
        raise ValueError(f"x must be non-negative, got {x}")
    return f"processed {x}"

# Get the State object by passing return_state=True
state_ok = always_succeeds(42, return_state=True)
print(f"Success run → state type: {type(state_ok).__name__}")
print(f"  is_completed(): {state_ok.is_completed()}")
print(f"  result():       {state_ok.result()}")

print()

# Catch the failure state without a try/except on the flow itself
state_fail = sometimes_fails(-5, return_state=True)
print(f"Failed run → state type: {type(state_fail).__name__}")
print(f"  is_failed():    {state_fail.is_failed()}")
print(f"  message:        {state_fail.message}")


### What just happened?

- **`return_state=True`** is a Prefect keyword argument accepted by every `@flow` call — it returns the `State` object instead of the raw result.
- `state.result()` unwraps the actual return value from a completed run.
- **Failed runs don't bubble up exceptions by default when you use `return_state=True`** — you get a `Failed` state object you can inspect programmatically.
- This is how monitoring systems poll Prefect: check `state.is_completed()` or `state.is_failed()` without wrapping everything in try/except.


## Step 5 · Flow configuration — name, description, retries

The `@flow` decorator accepts many configuration options. The most important for day-to-day use:

| Parameter | Type | What it does |
|---|---|---|
| `name` | `str` | Display name in the UI |
| `description` | `str` | Shown in UI sidebar |
| `retries` | `int` | Retry the whole flow N times on failure |
| `retry_delay_seconds` | `int` | Wait between retries |
| `timeout_seconds` | `int` | Kill the run after N seconds |
| `log_prints` | `bool` | Capture `print()` calls in Prefect logs |
| `version` | `str` | Tag runs with a version string |

`log_prints=True` is especially useful in Colab because you don't need `get_run_logger()` everywhere.


In [ ]:
from prefect import flow
import time

@flow(
    name="configured-etl",
    description="Demonstrates flow configuration options",
    retries=2,                  # retry up to 2 times if the flow raises
    retry_delay_seconds=1,      # wait 1 second between retries (use higher in production)
    log_prints=True,            # capture print() in Prefect logs — handy for quick scripts
    version="1.0.0",
)
def configured_etl(fail_on_attempt: int = 99):
    """
    A flow that intentionally fails on the specified attempt number
    to demonstrate automatic retries.
    """
    # flow_run_context gives metadata about the current run
    from prefect.context import get_run_context
    ctx = get_run_context()
    attempt = ctx.flow_run.run_count  # 1 on first try, 2 on first retry, etc.

    print(f"Attempt #{attempt} of configured_etl")

    if attempt < fail_on_attempt:
        print(f"Simulating failure on attempt {attempt}...")
        raise RuntimeError(f"Transient error on attempt {attempt}")

    print("Flow completed successfully!")
    return {"attempt": attempt, "status": "ok"}


# This will fail on attempt 1, retry, fail on attempt 2, retry, then succeed
result = configured_etl(fail_on_attempt=3)
print(f"\nFinal result: {result}")


### What just happened?

- The flow **automatically retried** without any try/except — Prefect handled it.
- **`retries=2`** means Prefect will retry the entire flow body up to 2 times after the first failure.
- **`get_run_context().flow_run.run_count`** tells you which attempt you're on — useful for idempotent logic.
- **`log_prints=True`** means `print()` statements appear in Prefect's structured log stream, not just stdout.
- In production, set `retry_delay_seconds` to something meaningful (30–300 s) to give upstream services time to recover.


## Step 6 · Running the Prefect UI (local server)

The Prefect UI gives you a graphical view of every flow run, its logs, state timeline, and duration.

**To start the server locally (in your terminal, not this notebook):**
```bash
prefect server start
```

Then open **http://127.0.0.1:4200** in your browser.

In the UI you can:
- See every run with its state (`Completed` / `Failed`)
- Click a run to see its full log stream
- Inspect the duration timeline
- Filter by flow name, state, date range
- See the task run graph (Day 2+)

> **Colab note:** The local server runs on your machine, not in Colab. To see your Colab runs in the UI, the notebook and UI must both point to the same Prefect API. By default, both use `http://127.0.0.1:4200/api` — this works if you run the notebook locally. In Colab cloud, runs are still tracked in the ephemeral in-process store and logged to stdout.


In [ ]:
# Check which Prefect API server this environment is pointed at
import prefect.settings as ps

# PREFECT_API_URL is the server endpoint; if blank, uses ephemeral in-process DB
api_url = ps.PREFECT_API_URL.value()
print(f"PREFECT_API_URL = {api_url or '(not set — using ephemeral in-process mode)'}")

# You can override it to point at your local server:
# import os
# os.environ["PREFECT_API_URL"] = "http://127.0.0.1:4200/api"

print()
print("To start the local server, run in your terminal:")
print("  prefect server start")
print("Then open: http://127.0.0.1:4200")


### What just happened?

- **`PREFECT_API_URL`** controls which server receives your run data. When it's unset, Prefect uses an in-process SQLite store — no server needed.
- In production you set this to a hosted Prefect Cloud URL or your self-hosted server.
- **Colab runs in ephemeral mode by default** — runs are tracked in-process and logged to stdout. Start the local server only when running notebooks on your own machine.
- All the code in this notebook works in both modes — the API is the same.


In [ ]:
# Challenge: Convert this plain function into a Prefect flow
#
# Requirements:
#   1. Add the @flow decorator with a meaningful name
#   2. Accept `n_users` as a parameter (default 3)
#   3. Use get_run_logger() (or log_prints=True) to log progress
#   4. Return a summary dict with keys: users_fetched, total_posts, ran_at
#   5. Handle the case where the API returns no posts gracefully
#
# Scaffold (remove TODOs and fill in):

import json
import urllib.request
from datetime import datetime
# TODO: import the right Prefect decorators

# TODO: add @flow decorator here
def multi_user_pipeline(n_users: int = 3):
    # TODO: set up logging
    results = []
    for user_id in range(1, n_users + 1):
        url = f"https://jsonplaceholder.typicode.com/posts?userId={user_id}"
        # TODO: fetch posts and log progress
        pass

    # TODO: return summary dict
    pass

# Uncomment and run once you've filled in the solution:
# result = multi_user_pipeline(n_users=3)
# print(result)


---
## Day 1 key concepts recap

| Concept | What to remember |
|---|---|
| `@flow` decorator | Wraps any Python function; minimum change to get state tracking |
| Flow run states | `Pending → Running → Completed / Failed / Crashed` |
| `get_run_logger()` | Returns a run-scoped logger; each run gets its own log stream |
| `return_state=True` | Pass to a flow call to get the `State` object instead of the raw result |
| `retries` / `retry_delay_seconds` | Automatically retry the whole flow on failure |
| `log_prints=True` | Capture `print()` in Prefect logs — great for quick scripts |
| `PREFECT_API_URL` | Controls which server receives run data; unset = ephemeral in-process |
| Prefect UI | `prefect server start` → `http://127.0.0.1:4200` — see every run, log, state |

> **Tip:** Prefect's biggest win over raw scripts is automatic state tracking — every run, whether it succeeds or fails, gets a persistent record you can query later.

---
## What's next
**Day 2** → Break your flow into `@task`-decorated units, understand the flow-vs-task distinction, and pass return values between tasks to create explicit data dependencies.

Mark Day 1 complete in your [tracker](../index.html).
